# OptiCrop Research Analytics - Phase 1
This notebook performs:
1. Dataset Validation
2. Data Quality Assessment
3. Descriptive Statistics
4. Missing Value Analysis
5. CSV & JSON Export


In [ ]:
# Install (Kaggle usually has these)
# !pip install missingno -q

import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

OUTPUT_DIR="outputs"
CSV_DIR=os.path.join(OUTPUT_DIR,"csv")
JSON_DIR=os.path.join(OUTPUT_DIR,"json")
CHART_DIR=os.path.join(OUTPUT_DIR,"charts")

for d in [OUTPUT_DIR,CSV_DIR,JSON_DIR,CHART_DIR]:
    os.makedirs(d,exist_ok=True)

REQUIRED_COLUMNS=[
    "N","P","K","temperature",
    "humidity","ph","rainfall","label"
]


In [41]:
# Load Dataset
DATASET_PATH="/kaggle/input/datasets/chitrakumari25/smart-agricultural-production-optimizing-engine/Crop_recommendation.csv"

df=pd.read_csv(DATASET_PATH)

print("Dataset Loaded Successfully")
display(df.head())


Dataset Loaded Successfully


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [ ]:
# Dataset Validation

validation=[]

validation.append({
    "Check":"Dataset Loaded",
    "Status":"PASS" if not df.empty else "FAIL",
    "Details":f"{len(df)} rows"
})

missing_cols=[c for c in REQUIRED_COLUMNS if c not in df.columns]

validation.append({
    "Check":"Required Columns",
    "Status":"PASS" if len(missing_cols)==0 else "FAIL",
    "Details":"None" if len(missing_cols)==0 else ",".join(missing_cols)
})

duplicate_cols=df.columns[df.columns.duplicated()].tolist()

validation.append({
    "Check":"Duplicate Columns",
    "Status":"PASS" if len(duplicate_cols)==0 else "FAIL",
    "Details":"None" if len(duplicate_cols)==0 else ",".join(duplicate_cols)
})

wrong_dtype=[]
for c in REQUIRED_COLUMNS[:-1]:
    if c in df.columns and not pd.api.types.is_numeric_dtype(df[c]):
        wrong_dtype.append(c)

validation.append({
    "Check":"Numeric Feature Types",
    "Status":"PASS" if len(wrong_dtype)==0 else "FAIL",
    "Details":"All Numeric" if len(wrong_dtype)==0 else ",".join(wrong_dtype)
})

validation_df=pd.DataFrame(validation)
display(validation_df)


In [ ]:
# Export Validation
validation_df.to_csv(f"{CSV_DIR}/validation_report.csv",index=False)

with open(f"{JSON_DIR}/validation_report.json","w") as f:
    json.dump(validation,f,indent=4)

print("Validation exported.")


In [ ]:
# Data Quality Assessment

quality={}

quality["Rows"]=len(df)
quality["Columns"]=len(df.columns)
quality["Memory_MB"]=round(df.memory_usage(deep=True).sum()/1024**2,3)
quality["Duplicate_Rows"]=int(df.duplicated().sum())
quality["Missing_Values"]=int(df.isna().sum().sum())
quality["Infinite_Values"]=int(np.isinf(df.select_dtypes(include=np.number)).sum().sum())

constant=df.columns[df.nunique()==1].tolist()

quality["Constant_Columns"]=len(constant)

quality_df=pd.DataFrame({
    "Metric":quality.keys(),
    "Value":quality.values()
})

display(quality_df)


In [ ]:
quality_df.to_csv(f"{CSV_DIR}/data_quality.csv",index=False)

with open(f"{JSON_DIR}/data_quality.json","w") as f:
    json.dump(quality,f,indent=4)

print("Data quality exported.")


In [ ]:
# Descriptive Statistics

numeric=df.select_dtypes(include=np.number)

stats=pd.DataFrame({
    "Mean":numeric.mean(),
    "Median":numeric.median(),
    "Mode":numeric.mode().iloc[0],
    "Minimum":numeric.min(),
    "Maximum":numeric.max(),
    "Range":numeric.max()-numeric.min(),
    "Variance":numeric.var(),
    "Std":numeric.std(),
    "Skewness":numeric.skew(),
    "Kurtosis":numeric.kurt(),
    "Missing":numeric.isna().sum()
})

for q in [0.25,0.5,0.75]:
    stats[f"Q{int(q*100)}"]=numeric.quantile(q)

stats=stats.round(4)

display(stats)


In [ ]:
stats.to_csv(f"{CSV_DIR}/descriptive_statistics.csv")

stats.to_json(f"{JSON_DIR}/descriptive_statistics.json",
              orient="index",
              indent=4)

print("Statistics exported.")


In [ ]:
# Missing Value Analysis

missing=pd.DataFrame({
    "Column":df.columns,
    "Missing":df.isna().sum().values,
    "Percentage":(df.isna().mean()*100).round(2).values
})

display(missing)

missing.to_csv(f"{CSV_DIR}/missing_value_summary.csv",index=False)

missing.to_json(f"{JSON_DIR}/missing_value_summary.json",
                orient="records",
                indent=4)


In [ ]:
# Missing Value Bar Chart

plt.figure(figsize=(10,4))
sns.barplot(data=missing,x="Column",y="Missing")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/missing_value_bar.png",dpi=300)
plt.show()


In [ ]:
# Missing Value Heatmap

plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(),cbar=False)
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/missing_value_heatmap.png",dpi=300)
plt.show()


In [ ]:
# Missingness Matrix

msno.matrix(df)
plt.savefig(f"{CHART_DIR}/missingno_matrix.png",dpi=300)
plt.show()


In [ ]:
# Dataset Summary

summary={
    "rows":len(df),
    "columns":len(df.columns),
    "numeric_columns":list(df.select_dtypes(include=np.number).columns),
    "categorical_columns":list(df.select_dtypes(exclude=np.number).columns),
    "target":"label" if "label" in df.columns else None,
    "exports":{
        "csv":CSV_DIR,
        "json":JSON_DIR,
        "charts":CHART_DIR
    }
}

with open(f"{JSON_DIR}/summary.json","w") as f:
    json.dump(summary,f,indent=4)

print(json.dumps(summary,indent=4))


## Phase 1 Complete

Generated outputs:

- validation_report
- data_quality
- descriptive_statistics
- missing_value_summary
- summary.json
- Missing value charts


# Phase 2 - Distribution, Crop, Soil, Climate & Correlation Analysis

In [ ]:

# Create output folders for phase 2
DIST_DIR = f"{CHART_DIR}/distributions"
os.makedirs(DIST_DIR, exist_ok=True)

numeric_cols = ["N","P","K","temperature","humidity","ph","rainfall"]


In [ ]:

# Distribution Analysis
distribution_summary=[]

for col in numeric_cols:
    distribution_summary.append({
        "Feature":col,
        "Mean":df[col].mean(),
        "Median":df[col].median(),
        "Std":df[col].std(),
        "Min":df[col].min(),
        "Max":df[col].max(),
        "Skewness":df[col].skew()
    })

    plt.figure(figsize=(8,4))
    sns.histplot(df[col],kde=True)
    plt.title(f"{col} Distribution")
    plt.tight_layout()
    plt.savefig(f"{DIST_DIR}/{col}_distribution.png",dpi=300)
    plt.close()

distribution_df=pd.DataFrame(distribution_summary).round(4)
display(distribution_df)

distribution_df.to_csv(f"{CSV_DIR}/distribution_summary.csv",index=False)
distribution_df.to_json(f"{JSON_DIR}/distribution_summary.json",orient="records",indent=4)


In [ ]:

# Crop Distribution

crop_dist=df["label"].value_counts().reset_index()
crop_dist.columns=["Crop","Count"]
crop_dist["Percentage"]=(crop_dist["Count"]/len(df)*100).round(2)

display(crop_dist)

crop_dist.to_csv(f"{CSV_DIR}/crop_distribution.csv",index=False)

plt.figure(figsize=(10,7))
sns.countplot(data=df,y="label",order=df["label"].value_counts().index)
plt.title("Crop Distribution")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/crop_distribution.png",dpi=300)
plt.show()


In [ ]:

# Soil Nutrient Analysis

soil_stats=df.groupby("label")[["N","P","K"]].agg(["mean","median","min","max","std"]).round(2)
display(soil_stats.head())

soil_stats.to_csv(f"{CSV_DIR}/soil_crop_statistics.csv")

for nutrient in ["N","P","K"]:
    plt.figure(figsize=(12,5))
    sns.boxplot(data=df,x="label",y=nutrient)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(f"{CHART_DIR}/{nutrient}_boxplot.png",dpi=300)
    plt.close()


In [ ]:

# Climate Analysis

climate_cols=["temperature","humidity","ph","rainfall"]

climate_stats=df.groupby("label")[climate_cols].mean().round(2)
display(climate_stats.head())

climate_stats.to_csv(f"{CSV_DIR}/climate_statistics.csv")

for feature in climate_cols:
    plt.figure(figsize=(12,5))
    sns.boxplot(data=df,x="label",y=feature)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(f"{CHART_DIR}/{feature}_boxplot.png",dpi=300)
    plt.close()


In [ ]:

# Correlation Analysis

pearson=df[numeric_cols].corr(method="pearson")
spearman=df[numeric_cols].corr(method="spearman")

pearson.to_csv(f"{CSV_DIR}/pearson_correlation.csv")
spearman.to_csv(f"{CSV_DIR}/spearman_correlation.csv")

plt.figure(figsize=(8,6))
sns.heatmap(pearson,annot=True,cmap="coolwarm",fmt=".2f")
plt.title("Pearson Correlation")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/pearson_heatmap.png",dpi=300)
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(spearman,annot=True,cmap="viridis",fmt=".2f")
plt.title("Spearman Correlation")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/spearman_heatmap.png",dpi=300)
plt.show()


In [ ]:

# Research Insights (Basic)

insights=[]

for col in numeric_cols:
    insights.append({
        "Feature":col,
        "Highest Mean Crop":df.groupby("label")[col].mean().idxmax(),
        "Lowest Mean Crop":df.groupby("label")[col].mean().idxmin(),
        "Overall Mean":round(df[col].mean(),2)
    })

insights_df=pd.DataFrame(insights)
display(insights_df)

insights_df.to_csv(f"{CSV_DIR}/research_insights.csv",index=False)
insights_df.to_json(f"{JSON_DIR}/research_insights.json",orient="records",indent=4)


## Phase 2 Complete

Outputs added:
- distribution_summary.csv
- crop_distribution.csv
- soil_crop_statistics.csv
- climate_statistics.csv
- pearson_correlation.csv
- spearman_correlation.csv
- research_insights.csv

Charts:
- Histograms with KDE
- Crop distribution
- Nutrient boxplots
- Climate boxplots
- Pearson/Spearman heatmaps


# Phase 3 - Advanced Research Analytics

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_selection import mutual_info_classif, f_classif
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.stats import zscore

ADV_DIR=f"{CHART_DIR}/advanced"
os.makedirs(ADV_DIR,exist_ok=True)

X=df[numeric_cols]
X_scaled=StandardScaler().fit_transform(X)


In [ ]:

# PCA Analysis
pca=PCA(n_components=2)
coords=pca.fit_transform(X_scaled)

pca_df=pd.DataFrame(coords,columns=["PC1","PC2"])
pca_df["label"]=df["label"]

pca_df.to_csv(f"{CSV_DIR}/pca_coordinates.csv",index=False)

variance=pd.DataFrame({
    "Component":["PC1","PC2"],
    "ExplainedVariance":pca.explained_variance_ratio_
})
variance.to_csv(f"{CSV_DIR}/pca_variance.csv",index=False)

plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df,x="PC1",y="PC2",hue="label",legend=False,s=35)
plt.title("PCA Projection")
plt.tight_layout()
plt.savefig(f"{ADV_DIR}/pca_scatter.png",dpi=300)
plt.show()


In [ ]:

# Crop Requirement Heatmap
crop_profile=df.groupby("label")[numeric_cols].mean().round(2)
crop_profile.to_csv(f"{CSV_DIR}/crop_profiles.csv")

plt.figure(figsize=(12,8))
sns.heatmap(crop_profile,cmap="YlGnBu")
plt.title("Crop Requirement Heatmap")
plt.tight_layout()
plt.savefig(f"{ADV_DIR}/crop_profile_heatmap.png",dpi=300)
plt.show()


In [ ]:

# KMeans Clustering
kmeans=KMeans(n_clusters=5,random_state=42,n_init=10)
clusters=kmeans.fit_predict(X_scaled)

cluster_df=pca_df.copy()
cluster_df["cluster"]=clusters
cluster_df.to_csv(f"{CSV_DIR}/cluster_results.csv",index=False)

plt.figure(figsize=(8,6))
sns.scatterplot(data=cluster_df,x="PC1",y="PC2",hue="cluster",palette="tab10",s=35)
plt.title("KMeans Clusters")
plt.tight_layout()
plt.savefig(f"{ADV_DIR}/kmeans_clusters.png",dpi=300)
plt.show()


In [ ]:

# Hierarchical Clustering
plt.figure(figsize=(12,5))
Z=linkage(X_scaled[:200],method="ward")
dendrogram(Z,no_labels=True)
plt.title("Hierarchical Clustering (First 200 Samples)")
plt.tight_layout()
plt.savefig(f"{ADV_DIR}/hierarchical_dendrogram.png",dpi=300)
plt.show()


In [ ]:

# Outlier Detection

z=np.abs(zscore(X))
outlier_mask=(z>3).any(axis=1)

outliers=df[outlier_mask].copy()
outliers.to_csv(f"{CSV_DIR}/outliers.csv",index=False)

summary=[]
for col in numeric_cols:
    q1=df[col].quantile(.25)
    q3=df[col].quantile(.75)
    iqr=q3-q1
    low=q1-1.5*iqr
    high=q3+1.5*iqr
    count=((df[col]<low)|(df[col]>high)).sum()
    summary.append({"Feature":col,"IQR_Outliers":int(count)})

out_summary=pd.DataFrame(summary)
out_summary.to_csv(f"{CSV_DIR}/outlier_summary.csv",index=False)
display(out_summary)


In [ ]:

# Feature Importance
y=df["label"]

mi=mutual_info_classif(X,y,random_state=42)
f_vals,p_vals=f_classif(X,y)

importance=pd.DataFrame({
    "Feature":numeric_cols,
    "MutualInformation":mi,
    "ANOVA_FScore":f_vals,
    "PValue":p_vals
}).sort_values("MutualInformation",ascending=False)

importance.to_csv(f"{CSV_DIR}/feature_importance.csv",index=False)
display(importance)

plt.figure(figsize=(8,4))
sns.barplot(data=importance,x="MutualInformation",y="Feature")
plt.title("Feature Importance (Mutual Information)")
plt.tight_layout()
plt.savefig(f"{ADV_DIR}/feature_importance.png",dpi=300)
plt.show()


In [ ]:

# Automated Research Observations

obs=[]

for feat in numeric_cols:
    means=df.groupby("label")[feat].mean()
    obs.append({
        "Observation":f"Highest average {feat}",
        "Crop":means.idxmax(),
        "Value":round(means.max(),2)
    })
    obs.append({
        "Observation":f"Lowest average {feat}",
        "Crop":means.idxmin(),
        "Value":round(means.min(),2)
    })

obs_df=pd.DataFrame(obs)
obs_df.to_csv(f"{CSV_DIR}/research_observations.csv",index=False)
obs_df.to_json(f"{JSON_DIR}/research_observations.json",orient="records",indent=4)

display(obs_df.head(10))


## Phase 3 Complete

New capabilities:
- PCA dimensionality reduction
- Crop profile heatmap
- K-Means clustering
- Hierarchical clustering
- Z-score & IQR outlier detection
- Mutual Information feature importance
- ANOVA F-score analysis
- Automated research observations

The notebook now produces advanced analytics suitable for integration into the OptiCrop Research Dashboard.


# Phase 4 - Final Report & Dashboard Export

In [33]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.9 MB/s eta 0:00:0000:0100:01


In [34]:

from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

REPORT_DIR=f"{OUTPUT_DIR}/reports"
os.makedirs(REPORT_DIR,exist_ok=True)
styles=getSampleStyleSheet()


In [35]:

# Dashboard Summary JSON

dashboard_summary={
    "dataset":{
        "rows":int(len(df)),
        "columns":int(len(df.columns)),
        "target":"label"
    },
    "files":{
        "csv":sorted(os.listdir(CSV_DIR)),
        "json":sorted(os.listdir(JSON_DIR)),
        "charts":sorted(os.listdir(CHART_DIR))
    },
    "quality":quality,
    "validation":validation
}

with open(f"{JSON_DIR}/dashboard_summary.json","w") as f:
    json.dump(dashboard_summary,f,indent=4)

print("Dashboard summary created.")


Dashboard summary created.


In [36]:

# Human-readable research summary

lines=[]

lines.append("OptiCrop Research Analytics Report")
lines.append("="*40)
lines.append(f"Dataset Size : {len(df)} rows x {len(df.columns)} columns")
lines.append(f"Number of Crops : {df['label'].nunique()}")
lines.append("")

lines.append("Highest Mean Values")
for feat in numeric_cols:
    grp=df.groupby("label")[feat].mean()
    lines.append(f"- {feat}: {grp.idxmax()} ({grp.max():.2f})")

summary_txt="\n".join(lines)

with open(f"{REPORT_DIR}/research_summary.txt","w") as f:
    f.write(summary_txt)

print(summary_txt)


OptiCrop Research Analytics Report
Dataset Size : 2200 rows x 8 columns
Number of Crops : 22

Highest Mean Values
- N: cotton (117.77)
- P: apple (134.22)
- K: grapes (200.11)
- temperature: papaya (33.72)
- humidity: coconut (94.84)
- ph: chickpea (7.34)
- rainfall: rice (236.18)


In [37]:

# PDF Report

doc=SimpleDocTemplate(f"{REPORT_DIR}/OptiCrop_Research_Report.pdf")
story=[]

story.append(Paragraph("<b>OptiCrop Research Analytics Report</b>",styles["Title"]))
story.append(Paragraph(f"Rows: {len(df)}",styles["BodyText"]))
story.append(Paragraph(f"Columns: {len(df.columns)}",styles["BodyText"]))
story.append(Paragraph(f"Crop Classes: {df['label'].nunique()}",styles["BodyText"]))

story.append(Paragraph("<br/><b>Data Quality</b>",styles["Heading2"]))
for k,v in quality.items():
    story.append(Paragraph(f"{k}: {v}",styles["BodyText"]))

story.append(Paragraph("<br/><b>Key Research Observations</b>",styles["Heading2"]))

for feat in numeric_cols:
    grp=df.groupby("label")[feat].mean()
    story.append(Paragraph(
        f"{feat}: Highest average observed for <b>{grp.idxmax()}</b> ({grp.max():.2f})",
        styles["BodyText"]
    ))

doc.build(story)
print("PDF Report Generated")


PDF Report Generated


In [38]:

# Final Export Manifest

manifest=[]

for folder in [CSV_DIR,JSON_DIR,CHART_DIR,REPORT_DIR]:
    for file in sorted(os.listdir(folder)):
        manifest.append({
            "Folder":os.path.basename(folder),
            "File":file
        })

manifest_df=pd.DataFrame(manifest)
display(manifest_df)

manifest_df.to_csv(f"{CSV_DIR}/export_manifest.csv",index=False)
manifest_df.to_json(f"{JSON_DIR}/export_manifest.json",orient="records",indent=4)

print("Export manifest created.")


,Folder,File
0,csv,climate_statistics.csv
1,csv,cluster_results.csv
2,csv,crop_distribution.csv
3,csv,crop_profiles.csv
4,csv,data_quality.csv
5,csv,descriptive_statistics.csv
6,csv,distribution_summary.csv
7,csv,feature_importance.csv
8,csv,missing_value_summary.csv
9,csv,outlier_summary.csv


Export manifest created.



# ✅ OptiCrop Research Analytics Pipeline Complete

The notebook now provides:

- Dataset validation
- Data quality assessment
- Descriptive statistics
- Missing value analysis
- Distribution analysis
- Crop distribution analysis
- Soil nutrient analysis
- Climate analysis
- Pearson & Spearman correlation
- PCA
- Crop profile heatmap
- K-Means clustering
- Hierarchical clustering
- Outlier detection
- Feature importance (Mutual Information & ANOVA)
- Automated research observations
- CSV, JSON and chart export
- Dashboard summary JSON
- PDF research report
- Export manifest

This notebook is ready to run on your sample agricultural dataset and provides outputs that can be consumed directly by your Flask dashboard.


In [39]:
!zip -r outputs.zip /kaggle/working/outputs

  adding: kaggle/working/outputs/ (stored 0%)
  adding: kaggle/working/outputs/json/ (stored 0%)
  adding: kaggle/working/outputs/json/data_quality.json (deflated 38%)
  adding: kaggle/working/outputs/json/validation_report.json (deflated 66%)
  adding: kaggle/working/outputs/json/distribution_summary.json (deflated 73%)
  adding: kaggle/working/outputs/json/export_manifest.json (deflated 87%)
  adding: kaggle/working/outputs/json/descriptive_statistics.json (deflated 76%)
  adding: kaggle/working/outputs/json/research_insights.json (deflated 76%)
  adding: kaggle/working/outputs/json/research_observations.json (deflated 81%)
  adding: kaggle/working/outputs/json/dashboard_summary.json (deflated 76%)
  adding: kaggle/working/outputs/json/missing_value_summary.json (deflated 83%)
  adding: kaggle/working/outputs/json/summary.json (deflated 56%)
  adding: kaggle/working/outputs/charts/ (stored 0%)
  adding: kaggle/working/outputs/charts/P_boxplot.png (deflated 22%)
  adding: kaggle/worki

In [ ]:
from IPython.display import FileLink
FileLink('outputs.zip')
